In [ ]:
# PART 1 : IMPORT LIBRARIES

import os
import random
import warnings
import numpy as np

import torch
import torch.nn as nn

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch Version :", torch.__version__)
print("Device          :", DEVICE)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATASET_PATH = "/content/drive/MyDrive"

In [ ]:
# LOAD DATASET
X = np.load(os.path.join(DATASET_PATH, "NEW_ROI_X.npy"))
Y = np.load(os.path.join(DATASET_PATH, "NEW_ROI_Y.npy"))

print("Dataset loaded successfully!\n")

print(f"Images Shape : {X.shape}")
print(f"Masks Shape  : {Y.shape}")

In [ ]:
print("Image Shape :", X.shape)
print("Mask Shape  :", Y.shape)

print("Image dtype :", X.dtype)
print("Mask dtype  :", Y.dtype)

print("Image Min/Max :", X.min(), X.max())
print("Mask Unique   :", np.unique(Y))

In [ ]:
import matplotlib.pyplot as plt
import random

plt.figure(figsize=(10, 10))

for i in range(5):
    idx = random.randint(0, len(X)-1)

    plt.subplot(5, 2, 2*i+1)
    plt.imshow(X[idx].squeeze(), cmap="gray")
    plt.title("Image")
    plt.axis("off")

    plt.subplot(5, 2, 2*i+2)
    plt.imshow(Y[idx].squeeze(), cmap="gray")
    plt.title("Mask")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# PART 2 : IMPORT LIBRARIES

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

In [ ]:
# TRAIN / VALIDATION / TEST SPLIT

X_train, X_temp, Y_train, Y_temp = train_test_split(
    X,
    Y,
    test_size=0.30,
    random_state=SEED,
    shuffle=True
)

X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp,
    Y_temp,
    test_size=0.50,
    random_state=SEED,
    shuffle=True
)

print(f"Training Images   : {len(X_train)}")
print(f"Validation Images : {len(X_val)}")
print(f"Testing Images    : {len(X_test)}")

In [ ]:
# CUSTOM PYTORCH DATASET
class BreastCancerDataset(Dataset):

    def __init__(self, images, masks):
        self.images = images
        self.masks = masks

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):

        image = self.images[index]
        mask = self.masks[index]

        # Convert NumPy -> Torch Tensor
        image = torch.from_numpy(image).float()
        mask = torch.from_numpy(mask).float()

        # Convert HWC -> CHW
        image = image.permute(2, 0, 1)
        mask = mask.permute(2, 0, 1)

        return image, mask

In [ ]:
# CREATE DATASETS

train_dataset = BreastCancerDataset(X_train, Y_train)
val_dataset = BreastCancerDataset(X_val, Y_val)
test_dataset = BreastCancerDataset(X_test, Y_test)

In [ ]:
# CREATE DATALOADERS

BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
# VERIFY DATALOADER

images, masks = next(iter(train_loader))

print(f"Images Shape : {images.shape}")
print(f"Masks Shape  : {masks.shape}")

print(f"Image dtype  : {images.dtype}")
print(f"Mask dtype   : {masks.dtype}")

print(f"Device       : {DEVICE}")

In [ ]:
# RESIDUAL BLOCK

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv_block = nn.Sequential(

            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels)

        )

        self.shortcut = nn.Sequential()

        if in_channels != out_channels:

            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels)
            )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = self.shortcut(x)
        x = self.conv_block(x)
        x = x + residual
        x = self.relu(x)
        return x

In [ ]:
# ENCODER BLOCK

class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.residual = ResidualBlock(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2)

    def forward(self, x):
        features = self.residual(x)
        pooled = self.pool(features)
        return features, pooled

In [ ]:
# DECODER BLOCK

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2
        )

        self.residual = ResidualBlock(
            out_channels + skip_channels,
            out_channels
        )

    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        x = self.residual(x)
        return x

In [ ]:
# ORIGINAL RESUNET

class ResUNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Encoder
        self.e1 = EncoderBlock(1, 32)
        self.e2 = EncoderBlock(32, 64)
        self.e3 = EncoderBlock(64, 128)
        self.e4 = EncoderBlock(128, 256)

        # Bridge
        self.bridge = ResidualBlock(256, 512)

        # Decoder
        self.d4 = DecoderBlock(512, 256, 256)
        self.d3 = DecoderBlock(256, 128, 128)
        self.d2 = DecoderBlock(128, 64, 64)
        self.d1 = DecoderBlock(64, 32, 32)

        # Output
        self.output = nn.Conv2d(
            32,
            1,
            kernel_size=1
        )

    def forward(self, x):

        s1, p1 = self.e1(x)
        s2, p2 = self.e2(p1)
        s3, p3 = self.e3(p2)
        s4, p4 = self.e4(p3)

        b = self.bridge(p4)

        d4 = self.d4(b, s4)
        d3 = self.d3(d4, s3)
        d2 = self.d2(d3, s2)
        d1 = self.d1(d2, s1)

        out = self.output(d1)

        return out

In [ ]:
# INITIALIZE MODEL

model = ResUNet().to(DEVICE)

print(model)

In [ ]:
# VERIFY MODEL
images, masks = next(iter(train_loader))
images = images.to(DEVICE)

with torch.no_grad():
    outputs = model(images)

print("Input Shape :", images.shape)
print("Output Shape:", outputs.shape)

In [ ]:
# MODEL PARAMETERS

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")

In [ ]:
# PART 4 : LOSS FUNCTIONS & METRICS

import torch.nn.functional as F

In [ ]:
# DICE COEFFICIENT

def dice_coefficient(pred, target, smooth=1e-6):
    """
    Computes Dice Score.

    Parameters:
        pred   : Model predictions
        target : Ground truth mask

    Returns:
        Dice coefficient
    """

    pred = pred.contiguous().view(-1)
    target = target.contiguous().view(-1)

    intersection = (pred * target).sum()

    dice = (2.0 * intersection + smooth) / (
        pred.sum() + target.sum() + smooth
    )

    return dice

In [ ]:
# INTERSECTION OVER UNION

def iou_score(pred, target, smooth=1e-6):
    """
    Computes IoU Score.
    """

    pred = pred.contiguous().view(-1)
    target = target.contiguous().view(-1)

    intersection = (pred * target).sum()

    union = pred.sum() + target.sum() - intersection

    iou = (intersection + smooth) / (union + smooth)

    return iou

In [ ]:
# DICE LOSS

class DiceLoss(nn.Module):

    def __init__(self):
        super().__init__()

    def forward(self, pred, target):

        dice = dice_coefficient(pred, target)

        return 1 - dice

In [ ]:
class BCEDiceLoss(nn.Module):

    def __init__(self):
        super().__init__()

        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()

    def forward(self, pred, target):

        bce_loss = self.bce(pred, target)

        # Apply sigmoid only for Dice computation
        pred = torch.sigmoid(pred)

        dice_loss = self.dice(pred, target)

        return bce_loss + dice_loss

In [ ]:
# LOSS FUNCTION

criterion = BCEDiceLoss()

print("Loss Function Initialized Successfully!")

In [ ]:
# METRICS HELPER FUNCTION
def calculate_metrics(outputs, masks):
    predictions = (torch.sigmoid(outputs) > 0.5).float()

    dice = dice_coefficient(predictions, masks)
    iou = iou_score(predictions, masks)

    return dice.item(), iou.item()

In [ ]:
# VERIFY LOSS FUNCTION

images, masks = next(iter(train_loader))

images = images.to(DEVICE)
masks = masks.to(DEVICE)

with torch.no_grad():
    outputs = model(images)

loss = criterion(outputs, masks)

dice, iou = calculate_metrics(outputs, masks)

print(f"Loss : {loss:.4f}")
print(f"Dice : {dice:.4f}")
print(f"IoU  : {iou:.4f}")

In [ ]:
# PART 5 : TRAINING SETUP

import copy
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.cuda.amp import GradScaler

In [ ]:
# TRAINING HYPERPARAMETERS

LEARNING_RATE = 1e-3
NUM_EPOCHS = 50

PATIENCE = 10

MODEL_PATH = "best_resunet.pth"

In [ ]:
# OPTIMIZER

optimizer = Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

In [ ]:
# LEARNING RATE SCHEDULER

scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

In [ ]:
# MIXED PRECISION

scaler = GradScaler(enabled=torch.cuda.is_available())

print("Mixed Precision:", torch.cuda.is_available())

In [ ]:
# EARLY STOPPING

class EarlyStopping:

    def __init__(self, patience=10):

        self.patience = patience
        self.counter = 0

        self.best_loss = float("inf")

        self.early_stop = False

    def __call__(self, val_loss):

        if val_loss < self.best_loss:

            self.best_loss = val_loss
            self.counter = 0

        else:

            self.counter += 1

            print(f"EarlyStopping Counter: {self.counter}/{self.patience}")

            if self.counter >= self.patience:

                self.early_stop = True

In [ ]:
# INITIALIZE EARLY STOPPING

early_stopping = EarlyStopping(
    patience=PATIENCE
)

In [ ]:
# TRAINING HISTORY

history = {

    "train_loss": [],
    "val_loss": [],

    "train_dice": [],
    "val_dice": [],

    "train_iou": [],
    "val_iou": []

}

In [ ]:
# SAVE BEST MODEL

best_val_loss = float("inf")

def save_best_model(model, val_loss):

    global best_val_loss

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(model.state_dict(), MODEL_PATH)

        print(f"Best model saved! Validation Loss: {val_loss:.4f}")

In [ ]:
# VERIFY TRAINING SETUP

print("Optimizer      :", optimizer.__class__.__name__)
print("Scheduler      :", scheduler.__class__.__name__)
print("Loss Function  :", criterion.__class__.__name__)
print("Device         :", DEVICE)

In [ ]:
# PART 6 : TRAINING LOOP

from tqdm.auto import tqdm

In [ ]:
# TRAIN ONE EPOCH

def train_one_epoch(model, loader, optimizer, criterion, scaler, device):

    model.train()

    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0

    progress_bar = tqdm(loader, desc="Training", leave=False)

    for images, masks in progress_bar:

        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        if device.type == "cuda":

            with torch.cuda.amp.autocast():

                outputs = model(images)
                loss = criterion(outputs, masks)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        else:

            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

        dice, iou = calculate_metrics(outputs, masks)

        running_loss += loss.item()
        running_dice += dice
        running_iou += iou

        progress_bar.set_postfix(
            Loss=f"{loss.item():.4f}",
            Dice=f"{dice:.4f}",
            IoU=f"{iou:.4f}"
        )

    return (
        running_loss / len(loader),
        running_dice / len(loader),
        running_iou / len(loader)
    )

In [ ]:
# VALIDATE ONE EPOCH

def validate_one_epoch(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0

    with torch.no_grad():

        progress_bar = tqdm(loader, desc="Validation", leave=False)

        for images, masks in progress_bar:

            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)

            loss = criterion(outputs, masks)

            dice, iou = calculate_metrics(outputs, masks)

            running_loss += loss.item()
            running_dice += dice
            running_iou += iou

            progress_bar.set_postfix(
                Loss=f"{loss.item():.4f}",
                Dice=f"{dice:.4f}",
                IoU=f"{iou:.4f}"
            )

    return (
        running_loss / len(loader),
        running_dice / len(loader),
        running_iou / len(loader)
    )

In [ ]:
# TRAIN MODEL

best_val_loss = float("inf")

for epoch in range(NUM_EPOCHS):

    print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}]")
    print("-" * 60)

    train_loss, train_dice, train_iou = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        scaler,
        DEVICE
    )

    val_loss, val_dice, val_iou = validate_one_epoch(
        model,
        val_loader,
        criterion,
        DEVICE
    )

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    history["train_dice"].append(train_dice)
    history["val_dice"].append(val_dice)

    history["train_iou"].append(train_iou)
    history["val_iou"].append(val_iou)

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")

    print(f"Train Dice : {train_dice:.4f}")
    print(f"Val Dice   : {val_dice:.4f}")

    print(f"Train IoU  : {train_iou:.4f}")
    print(f"Val IoU    : {val_iou:.4f}")

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(model.state_dict(), MODEL_PATH)

        print("Best model saved!")

    early_stopping(val_loss)

    if early_stopping.early_stop:

        print("\nEarly stopping triggered!")

        break

print("\nTraining Completed Successfully!")

In [ ]:
# ==========================================================
# PART 7 : LOAD BEST MODEL
# ==========================================================

model = ResUNet().to(DEVICE)

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))

model.eval()

print("Best model loaded successfully!")

In [ ]:
# ==========================================================
# TEST FUNCTION
# ==========================================================

def test_model(model, loader, criterion, device):

    model.eval()

    test_loss = 0.0
    test_dice = 0.0
    test_iou = 0.0

    with torch.no_grad():

        progress_bar = tqdm(loader, desc="Testing")

        for images, masks in progress_bar:

            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)

            loss = criterion(outputs, masks)

            dice, iou = calculate_metrics(outputs, masks)

            test_loss += loss.item()
            test_dice += dice
            test_iou += iou

    test_loss /= len(loader)
    test_dice /= len(loader)
    test_iou /= len(loader)

    return test_loss, test_dice, test_iou

In [ ]:
# ==========================================================
# EVALUATE MODEL
# ==========================================================

test_loss, test_dice, test_iou = test_model(
    model,
    test_loader,
    criterion,
    DEVICE
)

print("="*60)

print(f"Test Loss : {test_loss:.4f}")
print(f"Test Dice : {test_dice:.4f}")
print(f"Test IoU  : {test_iou:.4f}")

print("="*60)

In [ ]:
# ==========================================================
# SAVE RESULTS
# ==========================================================

results = {
    "Test Loss": test_loss,
    "Test Dice": test_dice,
    "Test IoU": test_iou
}

print(results)

In [ ]:
# PART 8 : VISUALIZATION

import matplotlib.pyplot as plt
import random

In [ ]:
# TRAINING CURVES

plt.figure(figsize=(18,5))

# Loss
plt.subplot(1,3,1)
plt.plot(history["train_loss"], label="Train")
plt.plot(history["val_loss"], label="Validation")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

# Dice
plt.subplot(1,3,2)
plt.plot(history["train_dice"], label="Train")
plt.plot(history["val_dice"], label="Validation")
plt.title("Dice Score")
plt.xlabel("Epoch")
plt.ylabel("Dice")
plt.legend()

# IoU
plt.subplot(1,3,3)
plt.plot(history["train_iou"], label="Train")
plt.plot(history["val_iou"], label="Validation")
plt.title("IoU")
plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# VISUALIZE TEST PREDICTIONS

model.eval()

NUM_SAMPLES = 10

indices = random.sample(range(len(test_dataset)), NUM_SAMPLES)

plt.figure(figsize=(12, NUM_SAMPLES*4))

for i, idx in enumerate(indices):

    image, mask = test_dataset[idx]

    input_tensor = image.unsqueeze(0).to(DEVICE)

    with torch.no_grad():

        output = model(input_tensor)

        prediction = (torch.sigmoid(output) > 0.5).float()

    image = image.squeeze().cpu().numpy()
    mask = mask.squeeze().cpu().numpy()
    prediction = prediction.squeeze().cpu().numpy()

    # Original Image
    plt.subplot(NUM_SAMPLES,3,3*i+1)
    plt.imshow(image, cmap="gray")
    plt.title("Original")
    plt.axis("off")

    # Ground Truth
    plt.subplot(NUM_SAMPLES,3,3*i+2)
    plt.imshow(mask, cmap="gray")
    plt.title("Ground Truth")
    plt.axis("off")

    # Prediction
    plt.subplot(NUM_SAMPLES,3,3*i+3)
    plt.imshow(prediction, cmap="gray")
    plt.title("Prediction")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# OVERLAY PREDICTIONS
# ==========================================================

NUM_SAMPLES = 5

indices = random.sample(range(len(test_dataset)), NUM_SAMPLES)

plt.figure(figsize=(15, NUM_SAMPLES*4))

for i, idx in enumerate(indices):

    image, mask = test_dataset[idx]

    image_np = image.squeeze().numpy()

    with torch.no_grad():
        output = model(image.unsqueeze(0).to(DEVICE))
        prediction = (torch.sigmoid(output) > 0.5).float().squeeze().cpu().numpy()

    # Original
    plt.subplot(NUM_SAMPLES,2,2*i+1)
    plt.imshow(image_np, cmap="gray")
    plt.title("Original")
    plt.axis("off")

    # Overlay
    plt.subplot(NUM_SAMPLES,2,2*i+2)
    plt.imshow(image_np, cmap="gray")
    plt.imshow(prediction, cmap="jet", alpha=0.4)
    plt.title("Prediction Overlay")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
def evaluate_model(model, loader, criterion, device):

    model.eval()

    total_loss = 0
    total_dice = 0
    total_iou = 0

    all_preds = []
    all_masks = []

    with torch.no_grad():

        for images, masks in loader:

            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)

            loss = criterion(outputs, masks)

            predictions = (torch.sigmoid(outputs) > 0.5).float()

            dice = dice_coefficient(predictions, masks)
            iou = iou_score(predictions, masks)

            total_loss += loss.item()
            total_dice += dice.item()
            total_iou += iou.item()

            all_preds.extend(predictions.cpu().numpy().flatten())
            all_masks.extend(masks.cpu().numpy().flatten())

    precision = precision_score(
        all_masks,
        all_preds,
        zero_division=0
    )

    recall = recall_score(
        all_masks,
        all_preds,
        zero_division=0
    )

    f1 = f1_score(
        all_masks,
        all_preds,
        zero_division=0
    )

    results = {
        "Loss": total_loss / len(loader),
        "Dice": total_dice / len(loader),
        "IoU": total_iou / len(loader),
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    }

    return results

In [ ]:
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

In [ ]:
train_results = evaluate_model(
    model,
    train_loader,
    criterion,
    DEVICE
)

val_results = evaluate_model(
    model,
    val_loader,
    criterion,
    DEVICE
)

test_results = evaluate_model(
    model,
    test_loader,
    criterion,
    DEVICE
)

In [ ]:
import pandas as pd

results_df = pd.DataFrame({
    "Metric": ["Loss", "Dice", "IoU", "Precision", "Recall", "F1 Score"],
    "Train": [
        train_results["Loss"],
        train_results["Dice"],
        train_results["IoU"],
        train_results["Precision"],
        train_results["Recall"],
        train_results["F1 Score"]
    ],
    "Validation": [
        val_results["Loss"],
        val_results["Dice"],
        val_results["IoU"],
        val_results["Precision"],
        val_results["Recall"],
        val_results["F1 Score"]
    ],
    "Test": [
        test_results["Loss"],
        test_results["Dice"],
        test_results["IoU"],
        test_results["Precision"],
        test_results["Recall"],
        test_results["F1 Score"]
    ]
})

# Round to 4 decimal places
results_df = results_df.round(4)

# Display as a nice table
display(results_df)

In [ ]:
#Save Model
torch.save(model.state_dict(), "ResUNet_CBIS_DDSM_Best.pth")